# 6. STAC Transactions with Authentication

So far you have only *read* from the STAC API. The [Transactions extension](https://github.com/stac-api-extensions/transactions) adds the endpoints for *writing*: creating, updating, and deleting collections and items over HTTP.

Writing to a catalog is a privileged operation, so this workshop puts an auth layer in front of the STAC API. In the docker-compose stack, requests go through [stac-auth-proxy](https://github.com/developmentseed/stac-auth-proxy) at `http://localhost:8084`:

- **Reads** (`GET`) are public
- **Writes** (`POST` / `PUT` / `DELETE`) require a bearer token

In this chapter you will get a token from the local identity provider, then walk through the full life cycle of a collection and an item: create it, read it back, update it, and delete it.

<div class="alert alert-block alert-info">
Every step asserts the status code it expects, so this notebook also doubles as an integration test you can run headless.
</div>

<div class="alert alert-block alert-warning">
<b>Note:</b> This chapter needs the local docker-compose auth stack (it reads <code>MOCK_OIDC_ENDPOINT</code>). It will not run against the hosted workshop deployment.
</div>

## 6.1 Get an access token

To write to the API you need to prove who you are. The stack runs a small [mock OIDC server](https://github.com/alukach/mock-oidc-server) that stands in for a real identity provider (like Auth0, Keycloak, or Cognito). You ask it for a token and it hands one back — no password required, since it is only for local testing.

The `stac_auth` helper module wraps that exchange so the notebook stays focused on STAC:

- `require_local_auth_stack()` checks that the auth endpoints are configured and returns them
- `get_mock_oidc_token()` requests a token with `stac:read` and `stac:write` scopes
- `auth_headers(token)` builds the `Authorization: Bearer <token>` header you attach to write requests

In [ ]:
import json
import time

import httpx

from stac_auth import auth_headers, get_mock_oidc_token, require_local_auth_stack

stac_api_endpoint, mock_oidc_endpoint = require_local_auth_stack()

write_token = get_mock_oidc_token()
write_headers = auth_headers(write_token)

print(f"STAC API:  {stac_api_endpoint}")
print(f"Mock OIDC: {mock_oidc_endpoint}")
print(f"Token (truncated): {write_token[:16]}...")

We will create one collection and one item during this walkthrough. Giving them a timestamped id keeps them unique so you can re-run the notebook without colliding with a previous run.

In [ ]:
run_id = int(time.time())
collection_id = f"tx-workshop-{run_id}"
item_id = f"tx-item-{run_id}"

print(f"Test collection: {collection_id}")
print(f"Test item:       {item_id}")

## 6.2 Confirm the transactions extension is enabled

The transactions endpoints only exist if the API was deployed with the extension turned on. As with any STAC capability, you can check the `/conformance` response before relying on it — look for conformance classes containing `transaction`.

<div class="alert alert-block alert-warning">
<b>Warning:</b> Never enable the transactions extension on a public API without an auth layer. Doing so lets anyone write to your catalog. That is exactly why this deployment sits behind stac-auth-proxy.
</div>

In [ ]:
conformance = httpx.get(f"{stac_api_endpoint}/conformance", timeout=10).json()

transaction_classes = [
    uri for uri in conformance["conformsTo"] if "transaction" in uri.lower()
]

print(json.dumps(transaction_classes, indent=2))
assert transaction_classes, "Expected STAC transaction conformance classes"

## 6.3 Create a collection

Items always live inside a collection, so we create the collection first. This is our first write, so it is a good place to see the auth boundary in action.

We `POST` the same collection twice:

1. **Without a token** — the proxy rejects it with `401`/`403`
2. **With our bearer token** — the write succeeds with `201`

Then we `GET` the collection back with no token, confirming reads stay public.

In [ ]:
collection = {
    "id": collection_id,
    "type": "Collection",
    "stac_version": "1.0.0",
    "description": "Temporary collection for the transactions + auth walkthrough.",
    "license": "CC-BY-4.0",
    "extent": {
        "spatial": {"bbox": [[-10, -10, 10, 10]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}

denied = httpx.post(f"{stac_api_endpoint}/collections", json=collection, timeout=10)
print(f"POST /collections without token -> {denied.status_code} (rejected)")
assert denied.status_code in (401, 403)

created = httpx.post(
    f"{stac_api_endpoint}/collections",
    headers=write_headers,
    json=collection,
    timeout=10,
)
print(f"POST /collections with token    -> {created.status_code} (created)")
assert created.status_code in (200, 201), created.text

In [ ]:
fetched = httpx.get(f"{stac_api_endpoint}/collections/{collection_id}", timeout=10)
print(f"GET /collections/{{id}} (no token) -> {fetched.status_code} (public read)")
assert fetched.status_code == 200
assert fetched.json()["id"] == collection_id

## 6.4 Add an item

Now we add an item to the collection by posting a GeoJSON `Feature` to the collection's `/items` endpoint. A STAC item needs at least an `id`, a `geometry`, a `bbox`, and a `datetime` in its `properties`.

Same auth rule applies: the unauthenticated `POST` is rejected, the authenticated one succeeds.

In [ ]:
item = {
    "id": item_id,
    "type": "Feature",
    "stac_version": "1.0.0",
    "collection": collection_id,
    "geometry": {"type": "Point", "coordinates": [0.5, 0.5]},
    "bbox": [0.5, 0.5, 0.5, 0.5],
    "properties": {"datetime": "2024-06-01T00:00:00Z"},
    "links": [],
    "assets": {},
}

items_url = f"{stac_api_endpoint}/collections/{collection_id}/items"

denied_item = httpx.post(items_url, json=item, timeout=10)
print(f"POST item without token -> {denied_item.status_code} (rejected)")
assert denied_item.status_code in (401, 403)

created_item = httpx.post(items_url, headers=write_headers, json=item, timeout=10)
print(f"POST item with token    -> {created_item.status_code} (created)")
assert created_item.status_code in (200, 201), created_item.text

## 6.5 Update an item

`PUT` **replaces** an item in full rather than patching a few fields. The safe pattern is therefore *get-then-modify*: fetch the current item, change what you need, and send the whole object back. This avoids accidentally dropping fields the server already stored.

Here we add a `description` property and `PUT` the modified item back.

In [ ]:
item_url = f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}"

stored_item = httpx.get(item_url, timeout=10).json()
stored_item["properties"]["description"] = "Updated by authenticated PUT"

denied_put = httpx.put(item_url, json=stored_item, timeout=10)
print(f"PUT item without token -> {denied_put.status_code} (rejected)")
assert denied_put.status_code in (401, 403)

updated_item = httpx.put(item_url, headers=write_headers, json=stored_item, timeout=10)
print(f"PUT item with token    -> {updated_item.status_code} (updated)")
assert updated_item.status_code == 200, updated_item.text
assert (
    updated_item.json()["properties"]["description"] == "Updated by authenticated PUT"
)

## 6.6 Delete an item

`DELETE` removes the item. Afterwards a `GET` for that item returns `404`, confirming it is gone.

In [ ]:
denied_delete = httpx.delete(item_url, timeout=10)
print(f"DELETE item without token -> {denied_delete.status_code} (rejected)")
assert denied_delete.status_code in (401, 403)

deleted_item = httpx.delete(item_url, headers=write_headers, timeout=10)
print(f"DELETE item with token    -> {deleted_item.status_code} (deleted)")
assert deleted_item.status_code in (200, 204)

missing_item = httpx.get(item_url, timeout=10)
print(f"GET deleted item          -> {missing_item.status_code} (not found)")
assert missing_item.status_code == 404

## 6.7 Clean up

Finally, delete the temporary collection so we leave the catalog as we found it. Deleting a collection is a write too, so it also needs the token.

You have now exercised the full transaction life cycle — create, read, update, delete — and seen the auth proxy enforce the read/write boundary at every step.

In [ ]:
collection_url = f"{stac_api_endpoint}/collections/{collection_id}"

denied_collection_delete = httpx.delete(collection_url, timeout=10)
print(
    f"DELETE collection without token -> {denied_collection_delete.status_code} (rejected)"
)
assert denied_collection_delete.status_code in (401, 403)

deleted_collection = httpx.delete(collection_url, headers=write_headers, timeout=10)
print(f"DELETE collection with token    -> {deleted_collection.status_code} (deleted)")
assert deleted_collection.status_code in (200, 204)

print("\nAll transaction + auth checks passed.")